In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import data_path
from src.spark_utils import get_spark
from src.feature_models.shingles import evaluate_shingles_pairs
import pyspark.sql.functions as F

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()
spark

In [ ]:
df = spark.read.csv(
    # str(data_path("near_duplicates_qqp_benchmark.csv")),
    str(data_path("quora_duplicate_questions.tsv")),
    header=True,
    sep="\\t",
    # sep=";",
    inferSchema=True,
)

'''
df = df.withColumn("is_near_duplicate", F.when(F.col("is_near_duplicate_semantic") == 1, True).otherwise(False))
df = df.drop("qid1", "qid2", "is_near_duplicate_semantic", "is_near_duplicate_syntax")
df = df.withColumnsRenamed({
    "Id": "ID",
    "question1": "data1",
    "question2": "data2",
})
'''
df = df.withColumnsRenamed({
    "Id": "ID",
    "question1": "data1",
    "question2": "data2",
})

shingles_metrics = []
shingles_df = df.alias("shingles_df")
shingles_df = evaluate_shingles_pairs(shingles_df, shingle_size=8, fingerprint_size=512, over_words=False)

shingles_df.show()

In [ ]:
spark.stop()